In [ ]:

# There is 10 prediction vectors per image (one per model) and each vector has 54 scores (one per class)
# There is one F1-score per target class per model (10 models x 18 classes = 180 scores)

# This script combines the 10 prediction vectors of 54 scores obtained for each image by :
# - getting the scores for each of the 18 target classes,
# - weighting the scores by the F1-scores per class per model,
# - summing and normalizing the scores to get the final vector of prediction per image.


In [1]:
import pandas as pd
import pickle
from collections import defaultdict
import numpy as np

In [2]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent.parent))
from config1 import MONITORING_DATA, classes54, classes18


In [ ]:
p_global_weights = MONITORING_DATA / "3.Input_Tables" / "globalweights_f1scores_18cl.csv"

p_pred= MONITORING_DATA / "2.PREDICTIONS_10models_"

li_trapsname = ["ANNU2019","ANNU2020","ANNU2021","ANNU2022","ANNU2023"]

mp_save = MONITORING_DATA / "PREDICTIONS_51slides_combined/"


In [4]:

class_keep =  ['Buxus', 'Cupressaceae',  'IndetBlurry', 'IndetCovered', 'Lycopodium', 'NonPollen', 'Olea',
               'Phillyrea', 'Pinaceae', 'Pistacia', 'Plantago', 'Poaceae', 'QuercusDeciduous', 'QuercusIlex', 'VitisF', 'VitisS']

class_sum_other = ['Other', 'Acacia', 'Acer', 'Alnus', 'Amarantaceae', 'Artemisia', 'Betulaceae', 'Brassicaceae', 'Carduus', 'Caryophyllaceae', 'Cichorioideae', 'Cyperaceae', 
               'Echium', 'Ericaceae', 'Fagus', 'Gallium', 'Juglans', 'Lamiaceae', 'Liliaceae', 'Moraceae', 'Morphotype1', 
               'Morphotype2', 'Myrtaceae', 'Platanus', 'PopulusSp', 'Ranunculaceae', 'Rhamnus', 'Rosaceae', 'Rumex', 'Salix', 'Sanguisorba', 'Tilia', 'Ulmus', 'Urtica',
                 'ViburnumSambucusTp', 'XanthiumAmbrosia']

class_sum_frax=['FraxinusExcelsior', 'FraxinusOrnus']

index_frax_excelsior = classes54.index("FraxinusExcelsior")
index_frax_ornus = classes54.index("FraxinusOrnus")




In [5]:
df_w = pd.read_csv(p_global_weights)
print(sum(df_w["counter"])) # 12540 env images 
df_w.head()


12540


,cval,classe,f1score,counter
0,fold1,Buxus,0.988235,84
1,fold2,Buxus,0.981818,83
2,fold3,Buxus,0.994012,83
3,fold5,Buxus,1.000000,83
4,fold4,Buxus,0.988235,84


In [6]:
weights_dict = {
    model: df_group.set_index("classe")["f1score"].to_dict()
    for model, df_group in df_w.groupby("cval")
}

sum_f1_by_model = df_w.groupby("cval")["f1score"].sum().to_dict()
print(weights_dict)



{'fold1': {'Buxus': 0.988235294117647, 'Cupressaceae': 0.956989247311828, 'Fraxinus': 0.87719298245614, 'IndetBlurry': 0.953488372093023, 'IndetCovered': 0.893617021276596, 'Lycopodium': 0.990033222591362, 'NonPollen': 0.937062937062937, 'Olea': 0.939393939393939, 'Other': 0.947890818858561, 'Phillyrea': 0.862745098039216, 'Pinaceae': 0.994818652849741, 'Pistacia': 0.909090909090909, 'Plantago': 0.985507246376812, 'Poaceae': 0.989473684210526, 'QuercusDeciduous': 0.782608695652174, 'QuercusIlex': 0.943661971830986, 'VitisF': 0.982456140350877, 'VitisS': 0.914285714285714}, 'fold10': {'Buxus': 0.976190476190476, 'Cupressaceae': 0.966666666666667, 'Fraxinus': 0.84, 'IndetBlurry': 0.98876404494382, 'IndetCovered': 0.888888888888889, 'Lycopodium': 1.0, 'NonPollen': 0.957746478873239, 'Olea': 0.984126984126984, 'Other': 0.944444444444444, 'Phillyrea': 0.872727272727273, 'Pinaceae': 1.0, 'Pistacia': 0.978723404255319, 'Plantago': 0.957746478873239, 'Poaceae': 0.989583333333333, 'QuercusDecid

In [7]:


def load_files_10models(trapsi):
    """
    input : prediction vectors generated for each image per each model, and the corresponding filenames
    predictions are vectors of 54 scores generated for each image per model (10 cross-validated models)

        output: dictionary associating for each model, the prediction vectors of each image, and their corresponding filenames
    """
    scores_d={}
    filenames_d={}
    
    for idx in [1,2,3,4,5,6,7,8,9,10]: 
        cval_id = f"fold{idx}"
        p_scores = str(p_pred) + f"/resnet152_{cval_id}/resnet152_{cval_id}_{trapsi}_pred_scores.pkl"
        p_filenames = str(p_pred) + f"/resnet152_{cval_id}/resnet152_{cval_id}_{trapsi}_filenames.pkl"
        with open(p_scores, 'rb') as f:
            scores_i = pickle.load(f)
        scores_d[cval_id]=scores_i
        with open(p_filenames, 'rb') as f:
            filenames_i = pickle.load(f)
        filenames_d[cval_id]=filenames_i

    return filenames_d, scores_d


In [8]:
# verification order is verified

for trapsi in li_trapsname:

    filenames, scores = load_files_10models(trapsi)
    reference = filenames["fold1"]
    same = all(filenames[f"fold{i}"] == reference for i in range(2, 11))
    print(same, 'sampling year:', trapsi, reference[0], '\n')
    print(scores["fold1"][2])

True sampling year: ANNU2019 W3_2019_A/W3_2019_A_03924.jpg 

[5.2130350e-29 3.5895590e-07 2.1357716e-11 2.7429482e-12 3.6507311e-09
 1.3312621e-08 7.8620069e-13 1.5224922e-06 3.7390599e-20 8.7886060e-20
 2.4931871e-18 9.2792737e-01 6.1878977e-09 2.1752172e-08 3.1870277e-15
 2.1658117e-19 1.8135615e-02 1.0300804e-04 4.4178203e-10 1.2481546e-09
 4.3803951e-04 1.8888219e-21 1.4488619e-10 6.4314363e-06 1.1589633e-16
 2.4622397e-09 2.1372277e-06 2.6028238e-15 9.6633421e-19 5.6505764e-06
 3.6718609e-10 3.0150878e-07 2.9289976e-03 8.3059393e-11 2.6268039e-03
 5.3383143e-08 3.7295657e-05 1.6668519e-05 1.5320572e-04 3.8273260e-02
 3.3502434e-03 5.2821569e-10 3.0681727e-05 3.4096563e-06 5.0013565e-04
 1.4179790e-03 1.8729976e-17 2.9251107e-19 4.7425727e-12 7.4742219e-11
 4.9288837e-11 2.5453251e-03 1.4954322e-03 7.8681348e-15]
True sampling year: ANNU2020 D2_2020_B/D2_2020_B_00429.jpg 

[0.00000000e+00 3.83874903e-19 6.61723680e-05 1.56996522e-20
 6.72459367e-23 9.99933720e-01 3.98852286e-31 2.9

In [9]:
saving="yes" #yes to dump the pickle files


tot_slides = []
tot_count = 0


for trapsi in li_trapsname:

    filenames, scores = load_files_10models(trapsi) # generates dict with predictions from 10 cross-validations filenames[cval_1] = list of strings, scores[cval_i] = list of vectors
    li_imgfiles_ref = filenames["fold1"]

    li_slides = sorted(set(el.split("/")[0] for el in li_imgfiles_ref))
    print('nb img : ', len(li_imgfiles_ref), "nb slides : ", len(li_slides))

    combined_vectors = {}

    for k, imgfile in enumerate(li_imgfiles_ref):

        if k % 4000 == 0: # tracking advancement
            print(k) 

        weightedscores_k = defaultdict(float)

        for idx in [1,2,3,4,5,6,7,8,9,10]: 
            cval_id = f"fold{idx}"

            vector_kj = scores[cval_id][k]
            img_test = filenames[cval_id][k]
            if img_test!=imgfile:
                print("error")
                break

            # 54 classes -> 18 target classes
            vector_kj_dict_target = {
                classes54[s]: float(score)
                for s, score in enumerate(vector_kj)
                if classes54[s] in class_keep
            }

            vector_kj_dict_target["Fraxinus"] = sum(
                float(score)
                for s, score in enumerate(vector_kj)
                if classes54[s] in class_sum_frax
            )

            vector_kj_dict_target["Other"] = sum(
                float(score)
                for s, score in enumerate(vector_kj)
                if classes54[s] in class_sum_other
            )

            weights_j = weights_dict.get(cval_id)
            sum_10w = sum_f1_by_model.get(cval_id)

            for classe_i in classes18:
                weight_ij = weights_j.get(classe_i)
                score_kij = vector_kj_dict_target[classe_i]

                weightedscore_ijk = (score_kij * weight_ij / sum_10w)
                weightedscores_k[classe_i] += weightedscore_ijk

        # Normalize
        sum_weightedscores = sum(weightedscores_k.values())
        norm_weightedscores_k = {key: float(value / sum_weightedscores) for key, value in weightedscores_k.items()}
        norm_weightedscores_k_vector = np.array([norm_weightedscores_k[c] for c in classes18])

        combined_vectors[str(imgfile)] = norm_weightedscores_k_vector

    tot_slides.append(li_slides)

    for slide_i in li_slides:

        combined_vectors_slidei = {k: v for k, v in combined_vectors.items() if k.split("/")[0] == slide_i}
        tot_count+=sum(sum(combined_vectors_slidei.values()))
        tot_i = len(combined_vectors_slidei)
        print("slide and nb of images : ", slide_i, tot_i)

        csvfilename = f"combinedpred_{slide_i}.pkl"
        if saving=="yes":
            with open(mp_save / csvfilename, 'wb') as f:
                pickle.dump(combined_vectors_slidei, f)
print("tot_count", tot_count) # should be 281518

nb img :  14285 nb slides :  5
0
4000
8000
12000
slide and nb of images :  D1_2019_A 1511
slide and nb of images :  W1_2019_A 2669
slide and nb of images :  W2_2019_A 1900
slide and nb of images :  W3_2019_A 4864
slide and nb of images :  W4_2019_A 3341
nb img :  31960 nb slides :  11
0
4000
8000
12000
16000
20000
24000
28000
slide and nb of images :  D1_2020_A 1131
slide and nb of images :  D1_2020_B 4419
slide and nb of images :  D2_2020_A 1199
slide and nb of images :  D2_2020_B 3829
slide and nb of images :  W1_2020_A 2156
slide and nb of images :  W1_2020_B 2577
slide and nb of images :  W2_2020_A 2982
slide and nb of images :  W3_2020_A 2384
slide and nb of images :  W3_2020_B 4737
slide and nb of images :  W4_2020_A 2514
slide and nb of images :  W4_2020_B 4032
nb img :  34284 nb slides :  9
0
4000
8000
12000
16000
20000
24000
28000
32000
slide and nb of images :  D1_2021_A 3836
slide and nb of images :  D2_2021_A 6622
slide and nb of images :  W1_2021_A 776
slide and nb of imag

In [10]:
print(li_slides)

['D2_2023_A', 'D2_2023_B', 'D3_2023_A', 'D3_2023_B', 'W1_2023_A', 'W1_2023_B', 'W2_2023_A', 'W2_2023_B', 'W2_2023_C', 'W3_2023_A', 'W3_2023_B', 'W4_2023_A', 'W4_2023_B', 'W4_2023_C']


In [ ]:
# end